# Load FX Rates from Excel

Reads the FX Rates Excel file and loads it into the `fx_rates` table.

**Prerequisites:** 
- Upload `FX Rates - Dec 2025.xlsx` to `Files/config/` in your lakehouse
- Source file: `docs/Source Documents/IBM data - HC/IBM data - HC/FX Rates - Dec 2025.xlsx`

In [ ]:
# ========== Establish Workspace Context ==========
try:
    spark.sql("SELECT 1")
    print("[INFO] Workspace context initialized")
except Exception as e:
    print(f"[WARNING] Workspace context issue: {str(e)}")

print("[FX RATES] Loading FX rates from Excel file")

In [ ]:
# ========== Imports ==========
from pyspark.sql import functions as F
import pandas as pd
from datetime import datetime

print("[INFO] Imports complete")

## Step 1: Read Excel File

In [ ]:
print("[STEP 1] Reading FX Rates Excel file...")

fx_file_relative = "FX Rates - Dec 2025.xlsx"
fx_file_path = f"Files/config/{fx_file_relative}"

try:
    # Use mssparkutils to find the file
    print(f"  Looking for: {fx_file_path}")
    
    # List config folder to find the file
    try:
        config_files = mssparkutils.fs.ls("Files/config/")
        print(f"  ✓ Found {len(config_files)} file(s) in Files/config/")
        
        # Find the FX rates file
        fx_file_full_path = None
        for f in config_files:
            print(f"    - {f.name}")
            if f.name.lower() == fx_file_relative.lower():
                fx_file_full_path = f.path
                print(f"  ✓ Located: {f.name}")
        
        if fx_file_full_path is None:
            raise FileNotFoundError(f"'{fx_file_relative}' not found in Files/config/")
        
        # Read Excel file using the full path
        pdf_fx = pd.read_excel(fx_file_full_path)
        
        print(f"\n✓ File loaded: {len(pdf_fx)} rows, {len(pdf_fx.columns)} columns")
        print(f"\nColumns: {pdf_fx.columns.tolist()}")
        print(f"\nFirst 10 rows:")
        print(pdf_fx.head(10))
        
    except Exception as e:
        print(f"  ✗ Error accessing Files/config/: {str(e)}")
        raise
    
except FileNotFoundError as e:
    print(f"✗ File not found: {str(e)}")
    print("\n[ACTION REQUIRED]:")
    print("1. In your Fabric lakehouse, go to Files section")
    print("2. Verify you're in the 'config' folder (shown in your screenshot)")
    print("3. Verify 'FX Rates - Dec 2025.xlsx' is visible there")
    print("4. If not, upload from: docs/Source Documents/IBM data - HC/IBM data - HC/")
    raise
except Exception as e:
    print(f"✗ Error reading file: {str(e)}")
    raise

## Step 2: Examine Structure and Transform

Based on typical FX rate files, the structure might be:
- **Option A**: Dates as columns, currency pairs as rows (wide format)
- **Option B**: Standard table format with columns: Date, From_Currency, To_Currency, Rate

We'll detect the structure and transform accordingly.

In [ ]:
print("[STEP 2] Analyzing structure...")

# Check if first column looks like currency pairs
first_col_name = pdf_fx.columns[0]
print(f"\nFirst column name: {first_col_name}")
print(f"First column sample values: {pdf_fx[first_col_name].head(5).tolist()}")

# Check if subsequent columns are dates
date_columns = []
for col in pdf_fx.columns[1:]:
    if pd.api.types.is_datetime64_any_dtype(pdf_fx[col]) or col.lower() in ['date', 'period']:
        date_columns.append(col)
    # Try parsing as date
    try:
        pd.to_datetime(col)
        date_columns.append(col)
    except:
        pass

if len(date_columns) > 0:
    print(f"\n✓ Detected date columns: {len(date_columns)}")
    print(f"  Format appears to be: Currency pairs (rows) x Dates (columns)")
    transformation_type = "WIDE_TO_LONG"
else:
    print(f"\n✓ Standard table format detected")
    transformation_type = "STANDARD"

print(f"\nTransformation type: {transformation_type}")

## Step 3: Transform to Target Schema

Target schema: `period, from_currency, to_currency, rate, effective_date, source, updated_at`

In [ ]:
print("[STEP 3] Transforming to target schema...")

if transformation_type == "WIDE_TO_LONG":
    print("  Pivoting wide format to long format...")
    
    # Assuming first column is currency pair like "USD/GBP"
    currency_pair_col = pdf_fx.columns[0]
    
    # Melt the dataframe to long format
    pdf_long = pdf_fx.melt(
        id_vars=[currency_pair_col],
        var_name='date',
        value_name='rate'
    )
    
    # Parse currency pairs (assuming format like "USD/GBP")
    pdf_long['from_currency'] = pdf_long[currency_pair_col].str.split('/').str[0].str.strip()
    pdf_long['to_currency'] = pdf_long[currency_pair_col].str.split('/').str[1].str.strip()
    
    # Parse date
    pdf_long['effective_date'] = pd.to_datetime(pdf_long['date']).dt.strftime('%Y-%m-%d')
    pdf_long['period'] = pd.to_datetime(pdf_long['date']).dt.strftime('%Y-%m')
    
    # Clean rate (remove any non-numeric characters)
    pdf_long['rate'] = pd.to_numeric(pdf_long['rate'], errors='coerce')
    
    # Add metadata
    pdf_long['source'] = 'FX Rates - Dec 2025.xlsx'
    
    # Select and rename columns
    pdf_final = pdf_long[[
        'period', 'from_currency', 'to_currency', 'rate', 
        'effective_date', 'source'
    ]]
    
    # Remove any rows with null rates
    pdf_final = pdf_final.dropna(subset=['rate'])
    
    print(f"  ✓ Transformed to {len(pdf_final)} rows")
    print(f"\n  Sample of transformed data:")
    print(pdf_final.head(10))
    
elif transformation_type == "STANDARD":
    print("  Processing standard table format...")
    
    # Map columns to target schema
    # Adjust these column names based on actual Excel structure
    pdf_final = pdf_fx.copy()
    
    # Expected columns in Excel: Date, From_Currency, To_Currency, Rate
    # Map to target schema
    column_mapping = {
        # Add actual column names from Excel here
        # Example: 'Date': 'effective_date', 'From': 'from_currency', ...
    }
    
    # If mapping needed, apply it
    if column_mapping:
        pdf_final = pdf_final.rename(columns=column_mapping)
    
    # Add derived columns
    if 'period' not in pdf_final.columns:
        pdf_final['period'] = pd.to_datetime(pdf_final['effective_date']).dt.strftime('%Y-%m')
    
    if 'source' not in pdf_final.columns:
        pdf_final['source'] = 'FX Rates - Dec 2025.xlsx'
    
    print(f"  ✓ Processed {len(pdf_final)} rows")
    print(f"\n  Sample of data:")
    print(pdf_final.head(10))

print(f"\n  Final row count: {len(pdf_final)}")
print(f"  Unique currency pairs: {pdf_final['from_currency'].nunique()} from, {pdf_final['to_currency'].nunique()} to")
print(f"  Unique periods: {pdf_final['period'].nunique()}")

## Step 4: Convert to Spark DataFrame

In [ ]:
print("[STEP 4] Converting to Spark DataFrame...")

# Convert to Spark
df_fx = spark.createDataFrame(pdf_final)

# Add timestamp
df_fx = df_fx.withColumn("updated_at", F.current_timestamp())

# Cast rate to DOUBLE
df_fx = df_fx.withColumn("rate", F.col("rate").cast("double"))

print(f"✓ Spark DataFrame created: {df_fx.count()} rows")
print(f"\nSchema:")
df_fx.printSchema()

print(f"\nSample data:")
df_fx.show(10, truncate=False)

## Step 5: Load into fx_rates Table

In [ ]:
print("[STEP 5] Loading into fx_rates table...")

# Check if table exists
try:
    existing_count = spark.table("fx_rates").count()
    print(f"  Existing rows in fx_rates: {existing_count}")
except:
    print("  ✗ fx_rates table does not exist")
    print("  [ACTION] Run nb_setup_tables.ipynb first to create the table")
    raise

# Load data
df_fx.write.mode("append").saveAsTable("fx_rates")

new_count = spark.table("fx_rates").count()
loaded_count = new_count - existing_count

print(f"\n✓ Loaded {loaded_count} new FX rates")
print(f"  Total rows in fx_rates: {new_count}")

## Step 6: Verify Loaded Data

In [ ]:
print("[STEP 6] Verifying loaded data...")

df_verify = spark.sql("""
    SELECT 
        period,
        COUNT(*) as rate_count,
        COUNT(DISTINCT from_currency) as from_currencies,
        COUNT(DISTINCT to_currency) as to_currencies,
        MIN(rate) as min_rate,
        MAX(rate) as max_rate
    FROM fx_rates
    GROUP BY period
    ORDER BY period DESC
""")

print("\nFX Rates Summary:")
df_verify.show(truncate=False)

print("\n[SUCCESS] FX rates loaded successfully")